In [1]:
import mediapipe
print(mediapipe.__version__)

1.0.1


In [ ]:
"""
Gesture-controlled Spotify.

Gestures
  Open palm (hold 0.5s) ........ arm / disarm (nothing else works while disarmed)
  Fist (hold briefly) .......... play / pause
  Index finger swipe LEFT ...... next track
  Index finger swipe RIGHT ..... previous track
  Thumb+finger pinch, drag ..... volume (drag up = louder, down = quieter)

Backends
  Windows: pip install pyautogui winsdk. Commands go straight to Spotify's media
           session (winsdk); without winsdk it falls back to global media keys.
  macOS:   AppleScript -> Spotify app    (no extra pip packages)

Keys in the camera window: 'q' quit.
"""
import os
import sys
import time
import queue
import threading
import subprocess
import urllib.request

import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

IS_WIN = sys.platform.startswith("win")
IS_MAC = sys.platform == "darwin"

HAVE_SMTC = False

if IS_WIN:
    import asyncio
    import ctypes
    from ctypes import wintypes
    import pyautogui
    pyautogui.PAUSE = 0

    # Optional: `pip install winsdk` lets us command Spotify's media session directly,
    # regardless of which app most recently grabbed the media keys.
    try:
        from winsdk.windows.media.control import (
            GlobalSystemMediaTransportControlsSessionManager as _SMTC,
        )
        HAVE_SMTC = True
    except Exception:
        HAVE_SMTC = False

    _user32 = ctypes.windll.user32
    _kernel32 = ctypes.windll.kernel32
    _kernel32.OpenProcess.restype = wintypes.HANDLE
    _kernel32.OpenProcess.argtypes = [wintypes.DWORD, wintypes.BOOL, wintypes.DWORD]
    _kernel32.CloseHandle.argtypes = [wintypes.HANDLE]
    _kernel32.QueryFullProcessImageNameW.argtypes = [
        wintypes.HANDLE, wintypes.DWORD, wintypes.LPWSTR, ctypes.POINTER(wintypes.DWORD)]

    def _find_spotify_hwnd():
        """Find Spotify's main window by process name (its title is the song name)."""
        found = []

        @ctypes.WINFUNCTYPE(wintypes.BOOL, wintypes.HWND, wintypes.LPARAM)
        def _cb(hwnd, _lparam):
            if not _user32.IsWindowVisible(hwnd) or _user32.GetWindowTextLengthW(hwnd) == 0:
                return True
            pid = wintypes.DWORD()
            _user32.GetWindowThreadProcessId(hwnd, ctypes.byref(pid))
            h = _kernel32.OpenProcess(0x1000, False, pid.value)  # QUERY_LIMITED_INFORMATION
            if h:
                buf = ctypes.create_unicode_buffer(260)
                size = wintypes.DWORD(260)
                if _kernel32.QueryFullProcessImageNameW(h, 0, buf, ctypes.byref(size)):
                    if buf.value.lower().endswith("spotify.exe"):
                        found.append(hwnd)
                _kernel32.CloseHandle(h)
            return True

        _user32.EnumWindows(_cb, 0)
        return found[0] if found else None

    def focus_spotify_window():
        hwnd = _find_spotify_hwnd()
        if not hwnd:
            print("Spotify window not found (is it running, not minimized to tray?)")
            return False
        if _user32.IsIconic(hwnd):
            _user32.ShowWindow(hwnd, 9)               # SW_RESTORE
        # Tapping Alt lets us bypass Windows' foreground-lock restriction
        _user32.keybd_event(0x12, 0, 0, 0)
        _user32.keybd_event(0x12, 0, 2, 0)
        _user32.SetForegroundWindow(hwnd)
        return True

    def minimize_spotify_window():
        hwnd = _find_spotify_hwnd()
        if hwnd:
            _user32.ShowWindow(hwnd, 6)   # SW_MINIMIZE

    async def _smtc_call(method):
        mgr = await _SMTC.request_async()
        for s in mgr.get_sessions():
            if "spotify" in s.source_app_user_model_id.lower():
                return await getattr(s, method)()
        return None                                   # no Spotify session found

    def spotify_transport(method, fallback_key):
        """Command Spotify's own media session; fall back to global media keys."""
        if HAVE_SMTC:
            try:
                if asyncio.run(_smtc_call(method)) is not None:
                    return
            except Exception as e:
                print(f"Media-session control failed ({e}); using media key.")
        pyautogui.press(fallback_key)

# ---------------- Config ----------------
MODEL_FILENAME = "hand_landmarker.task"
MODEL_URL = ("https://storage.googleapis.com/mediapipe-models/hand_landmarker/"
             "hand_landmarker/float16/1/hand_landmarker.task")

PALM_HOLD = 0.5           # seconds to hold open palm to arm/disarm
FIST_HOLD = 0.3           # seconds to hold fist to toggle play/pause
PINCH_THRESHOLD = 0.35    # thumb-index distance / hand size
SWIPE_DIST = 0.25         # fraction of frame width the fingertip must travel
SWIPE_WINDOW = 0.40       # ...within this many seconds
SWIPE_COOLDOWN = 0.8      # seconds between track changes
VOLUME_GAIN = 150         # percent of volume per full frame-height of hand travel
VOLUME_DEADZONE = 0.002   # ignore tiny vertical jitter
FOCUS_SPOTIFY_ON_ARM = True  # bring the Spotify window to the front when you arm
MINIMIZE_SPOTIFY_ON_DISARM = True  # minimize the Spotify window when you disarm

# ---------------- Spotify / OS control ----------------
class Player:
    """Runs slow actions (osascript / subprocess) on a worker thread so video never stutters."""

    def __init__(self):
        self.q = queue.Queue()
        threading.Thread(target=self._worker, daemon=True).start()

    def _worker(self):
        while True:
            fn, args = self.q.get()
            try:
                fn(*args)
            except Exception as e:
                print(f"Action failed: {e}")

    def _run(self, fn, *args):
        self.q.put((fn, args))

    # --- low-level backends ---
    @staticmethod
    def _osa(cmd):
        out = subprocess.run(["osascript", "-e", f'tell application "Spotify" to {cmd}'],
                             capture_output=True, text=True)
        return out.stdout.strip()

    @staticmethod
    def _ctl(*args):
        subprocess.run(["playerctl", *args], capture_output=True)

    # --- public actions ---
    def play_pause(self):
        if IS_WIN: self._run(spotify_transport, "try_toggle_play_pause_async", "playpause")
        elif IS_MAC: self._run(self._osa, "playpause")
        else: self._run(self._ctl, "play-pause")

    def next_track(self):
        if IS_WIN: self._run(spotify_transport, "try_skip_next_async", "nexttrack")
        elif IS_MAC: self._run(self._osa, "next track")
        else: self._run(self._ctl, "next")

    def previous_track(self):
        if IS_WIN: self._run(spotify_transport, "try_skip_previous_async", "prevtrack")
        elif IS_MAC: self._run(self._osa, "previous track")
        else: self._run(self._ctl, "previous")

    def focus_spotify(self):
        """Bring Spotify to the foreground (used when arming)."""
        if IS_WIN: 
            self._run(focus_spotify_window)
        elif IS_MAC: self._run(self._osa, "activate")
    

    def minimize_spotify(self):
        """Minimize Spotify (used when disarming)."""
        if IS_WIN:
            self._run(minimize_spotify_window)
        elif IS_MAC:  # macOS hides the app
            self._run(subprocess.run, ["osascript", "-e",
                'tell application "System Events" to set visible of process "Spotify" to false'])

            
    # Volume: macOS sets Spotify's absolute volume; others step the volume.

    @property
    def absolute_volume(self):
        return IS_MAC

    def get_volume(self):
        try:
            return int(self._osa("sound volume"))
        except Exception:
            return 50

    def set_volume(self, pct):                 # macOS only
        if self.q.qsize() < 2:                 # drop updates if we're backed up
            self._run(self._osa, f"set sound volume to {int(pct)}")

    def volume_steps(self, n):                 # Windows / Linux; n>0 up, n<0 down
        if IS_WIN:
            self._run(pyautogui.press, "volumeup" if n > 0 else "volumedown", abs(n))
        else:
            self._run(self._ctl, "volume", f"{0.05 * abs(n):.2f}{'+' if n > 0 else '-'}")


# ---------------- Hand detection ----------------
def ensure_model_exists():
    if not os.path.exists(MODEL_FILENAME):
        print("Downloading hand landmark model...")
        urllib.request.urlretrieve(MODEL_URL, MODEL_FILENAME)


class HandDetector:
    def __init__(self):
        options = vision.HandLandmarkerOptions(
            base_options=python.BaseOptions(model_asset_path=MODEL_FILENAME),
            running_mode=vision.RunningMode.VIDEO,
            num_hands=1,
            min_hand_detection_confidence=0.7,
            min_hand_presence_confidence=0.7,
        )
        self.detector = vision.HandLandmarker.create_from_options(options)

    def process(self, frame, ts_ms):
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        res = self.detector.detect_for_video(img, ts_ms)
        return res.hand_landmarks[0] if res.hand_landmarks else None


def dist(a, b):
    return float(np.hypot(a.x - b.x, a.y - b.y))


def finger_states(lm):
    """[index, middle, ring, pinky] extended? (tip farther from wrist than PIP joint)."""
    w = lm[0]
    return [dist(lm[t], w) > dist(lm[p], w) for t, p in [(8, 6), (12, 10), (16, 14), (20, 18)]]


def classify(lm):
    idx, mid, ring, pinky = finger_states(lm)
    hand_size = dist(lm[0], lm[9]) or 1e-6
    close = dist(lm[4], lm[8]) / hand_size < PINCH_THRESHOLD

    if close and (mid or ring or pinky):   # thumb+index touching, hand not a fist
        return "pinch"
    if idx and mid and ring and pinky:
        return "palm"
    if idx and not (mid or ring or pinky):
        return "point"
    if not (idx or mid or ring or pinky):
        return "fist"
    return "other"


# ---------------- Gesture logic ----------------
class GestureController:
    def __init__(self, player):
        self.player = player
        self.armed = False
        self.palm_start = None
        self.palm_locked = False     # must drop palm before it can toggle again
        self.fist_start = None
        self.fist_fired = False
        self.swipe_hist = []
        self.last_swipe = 0
        self.pinch_prev_y = None
        self.vol_start = 50
        self.vol_accum = 0.0
        self.palm_progress = 0.0
        self.status = ""
        self.status_time = 0

    def flash(self, text):
        self.status, self.status_time = text, time.time()

    def reset_all(self):
        self.palm_start = None
        self.palm_locked = False
        self.fist_start = None
        self.fist_fired = False
        self.swipe_hist.clear()
        self.pinch_prev_y = None
        self.palm_progress = 0.0

    def update(self, lm):
        now = time.time()
        mode = classify(lm) if lm else "none"

        # Reset per-gesture state when leaving that gesture
        if mode != "palm":
            self.palm_start, self.palm_locked, self.palm_progress = None, False, 0.0
        if mode != "fist":
            self.fist_start, self.fist_fired = None, False
        if mode != "point":
            self.swipe_hist.clear()
        if mode != "pinch":
            self.pinch_prev_y = None

        # --- Arm / disarm (always active) ---
        if mode == "palm" and not self.palm_locked:
            if self.palm_start is None:
                self.palm_start = now
            self.palm_progress = min((now - self.palm_start) / PALM_HOLD, 1.0)
            if self.palm_progress >= 1.0:
                self.armed = not self.armed
                self.palm_locked = True
                self.palm_progress = 0.0
                self.flash("ARMED" if self.armed else "DISARMED")
                if self.armed and FOCUS_SPOTIFY_ON_ARM:
                    self.player.focus_spotify()
                elif not self.armed and MINIMIZE_SPOTIFY_ON_DISARM:
                    self.player.minimize_spotify()
            return

        if not self.armed:
            return

        # --- Play / pause ---
        if mode == "fist":
            if self.fist_start is None:
                self.fist_start = now
            elif not self.fist_fired and now - self.fist_start >= FIST_HOLD:
                self.player.play_pause()
                self.fist_fired = True
                self.flash("PLAY / PAUSE")

        # --- Swipe tracks (index finger only) ---
        elif mode == "point":
            tip = lm[8]
            self.swipe_hist.append((now, tip.x, tip.y))
            self.swipe_hist = [h for h in self.swipe_hist if now - h[0] <= SWIPE_WINDOW]
            dx = self.swipe_hist[-1][1] - self.swipe_hist[0][1]
            dy = self.swipe_hist[-1][2] - self.swipe_hist[0][2]
            if (abs(dx) > SWIPE_DIST and abs(dy) < abs(dx) * 0.7
                    and now - self.last_swipe > SWIPE_COOLDOWN):
                # Frame is mirrored, so dx<0 = your hand moved to your left
                if dx < 0:
                    self.player.next_track()
                    self.flash("NEXT TRACK")
                else:
                    self.player.previous_track()
                    self.flash("PREVIOUS TRACK")
                self.last_swipe = now
                self.swipe_hist.clear()

        # --- Volume (pinch + drag) ---
        elif mode == "pinch":
            y = lm[8].y
            if self.pinch_prev_y is None:
                self.pinch_prev_y = y
                self.vol_accum = 0.0
                if self.player.absolute_volume:
                    self.vol_start = self.player.get_volume()
            else:
                dy = self.pinch_prev_y - y          # up = positive
                self.pinch_prev_y = y
                if abs(dy) > VOLUME_DEADZONE:
                    self.vol_accum += dy * VOLUME_GAIN
                    if self.player.absolute_volume:
                        v = int(np.clip(self.vol_start + self.vol_accum, 0, 100))
                        self.player.set_volume(v)
                        self.flash(f"VOLUME {v}%")
                    else:
                        steps = int(self.vol_accum / 2)   # ~2% per OS volume step
                        if steps:
                            self.player.volume_steps(steps)
                            self.vol_accum -= steps * 2
                            self.flash("VOLUME UP" if steps > 0 else "VOLUME DOWN")


# ---------------- Main loop ----------------
def main():
    ensure_model_exists()
    cap = cv2.VideoCapture(0)
    detector = HandDetector()
    ctrl = GestureController(Player())
    print("Hold an open palm for 0.5s to ARM. Press 'q' in the camera window to quit.")

    while cap.isOpened():
        ok, frame = cap.read()
        if not ok:
            break
        frame = cv2.flip(frame, 1)
        lm = detector.process(frame, int(time.time() * 1000))
        ctrl.update(lm)

        h, w, _ = frame.shape
        if lm:
            for p in lm:
                cv2.circle(frame, (int(p.x * w), int(p.y * h)), 4, (255, 0, 0), cv2.FILLED)

        color = (0, 200, 0) if ctrl.armed else (0, 0, 255)
        cv2.putText(frame, "ARMED" if ctrl.armed else "DISARMED", (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
        if ctrl.palm_progress > 0:  # hold-to-toggle progress bar
            cv2.rectangle(frame, (10, 55), (210, 70), (255, 255, 255), 1)
            cv2.rectangle(frame, (10, 55), (10 + int(200 * ctrl.palm_progress), 70), color, cv2.FILLED)
        if time.time() - ctrl.status_time < 1.5:
            cv2.putText(frame, ctrl.status, (10, h - 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)

        cv2.imshow("Spotify Gesture Control", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

# ---------------- Spotify / OS control ----------------
class Player:
    """Runs slow actions (osascript / subprocess) on a worker thread so video never stutters."""

    def __init__(self):
        self.q = queue.Queue()
        threading.Thread(target=self._worker, daemon=True).start()

    def _worker(self):
        while True:
            fn, args = self.q.get()
            try:
                fn(*args)
            except Exception as e:
                print(f"Action failed: {e}")

    def _run(self, fn, *args):
        self.q.put((fn, args))

    # --- low-level backends ---
    @staticmethod
    def _osa(cmd):
        out = subprocess.run(["osascript", "-e", f'tell application "Spotify" to {cmd}'],
                             capture_output=True, text=True)
        return out.stdout.strip()

    @staticmethod
    def _ctl(*args):
        subprocess.run(["playerctl", *args], capture_output=True)

    # --- public actions ---
    def play_pause(self):
        if IS_WIN: self._run(spotify_transport, "try_toggle_play_pause_async", "playpause")
        elif IS_MAC: self._run(self._osa, "playpause")
        else: self._run(self._ctl, "play-pause")

    def next_track(self):
        if IS_WIN: self._run(spotify_transport, "try_skip_next_async", "nexttrack")
        elif IS_MAC: self._run(self._osa, "next track")
        else: self._run(self._ctl, "next")

    def previous_track(self):
        if IS_WIN: self._run(spotify_transport, "try_skip_previous_async", "prevtrack")
        elif IS_MAC: self._run(self._osa, "previous track")
        else: self._run(self._ctl, "previous")

    def focus_spotify(self):
        """Bring Spotify to the foreground (used when arming)."""
        if IS_WIN: self._run(focus_spotify_window)
        elif IS_MAC: self._run(self._osa, "activate")

    # Volume: macOS sets Spotify's absolute volume; others step the volume.
    @property
    def absolute_volume(self):
        return IS_MAC

    def get_volume(self):
        try:
            return int(self._osa("sound volume"))
        except Exception:
            return 50

    def set_volume(self, pct):                 # macOS only
        if self.q.qsize() < 2:                 # drop updates if we're backed up
            self._run(self._osa, f"set sound volume to {int(pct)}")

    def volume_steps(self, n):                 # Windows / Linux; n>0 up, n<0 down
        if IS_WIN:
            self._run(pyautogui.press, "volumeup" if n > 0 else "volumedown", abs(n))
        else:
            self._run(self._ctl, "volume", f"{0.05 * abs(n):.2f}{'+' if n > 0 else '-'}")


# ---------------- Hand detection ----------------
def ensure_model_exists():
    if not os.path.exists(MODEL_FILENAME):
        print("Downloading hand landmark model...")
        urllib.request.urlretrieve(MODEL_URL, MODEL_FILENAME)


class HandDetector:
    def __init__(self):
        options = vision.HandLandmarkerOptions(
            base_options=python.BaseOptions(model_asset_path=MODEL_FILENAME),
            running_mode=vision.RunningMode.VIDEO,
            num_hands=1,
            min_hand_detection_confidence=0.7,
            min_hand_presence_confidence=0.7,
        )
        self.detector = vision.HandLandmarker.create_from_options(options)

    def process(self, frame, ts_ms):
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        res = self.detector.detect_for_video(img, ts_ms)
        return res.hand_landmarks[0] if res.hand_landmarks else None


def dist(a, b):
    return float(np.hypot(a.x - b.x, a.y - b.y))


def finger_states(lm):
    """[index, middle, ring, pinky] extended? (tip farther from wrist than PIP joint)."""
    w = lm[0]
    return [dist(lm[t], w) > dist(lm[p], w) for t, p in [(8, 6), (12, 10), (16, 14), (20, 18)]]


def classify(lm):
    idx, mid, ring, pinky = finger_states(lm)
    hand_size = dist(lm[0], lm[9]) or 1e-6
    close = dist(lm[4], lm[8]) / hand_size < PINCH_THRESHOLD

    if close and (mid or ring or pinky):   # thumb+index touching, hand not a fist
        return "pinch"
    if idx and mid and ring and pinky:
        return "palm"
    if idx and not (mid or ring or pinky):
        return "point"
    if not (idx or mid or ring or pinky):
        return "fist"
    return "other"


# ---------------- Gesture logic ----------------
class GestureController:
    def __init__(self, player):
        self.player = player
        self.armed = False
        self.palm_start = None
        self.palm_locked = False     # must drop palm before it can toggle again
        self.fist_start = None
        self.fist_fired = False
        self.swipe_hist = []
        self.last_swipe = 0
        self.pinch_prev_y = None
        self.vol_start = 50
        self.vol_accum = 0.0
        self.palm_progress = 0.0
        self.status = ""
        self.status_time = 0

    def flash(self, text):
        self.status, self.status_time = text, time.time()

    def reset_all(self):
        self.palm_start = None
        self.palm_locked = False
        self.fist_start = None
        self.fist_fired = False
        self.swipe_hist.clear()
        self.pinch_prev_y = None
        self.palm_progress = 0.0

    def update(self, lm):
        now = time.time()
        mode = classify(lm) if lm else "none"

        # Reset per-gesture state when leaving that gesture
        if mode != "palm":
            self.palm_start, self.palm_locked, self.palm_progress = None, False, 0.0
        if mode != "fist":
            self.fist_start, self.fist_fired = None, False
        if mode != "point":
            self.swipe_hist.clear()
        if mode != "pinch":
            self.pinch_prev_y = None

        # --- Arm / disarm (always active) ---
        if mode == "palm" and not self.palm_locked:
            if self.palm_start is None:
                self.palm_start = now
            self.palm_progress = min((now - self.palm_start) / PALM_HOLD, 1.0)
            if self.palm_progress >= 1.0:
                self.armed = not self.armed
                self.palm_locked = True
                self.palm_progress = 0.0
                self.flash("ARMED" if self.armed else "DISARMED")
                if self.armed and FOCUS_SPOTIFY_ON_ARM:
                    self.player.focus_spotify()
            return

        if not self.armed:
            return

        # --- Play / pause ---
        if mode == "fist":
            if self.fist_start is None:
                self.fist_start = now
            elif not self.fist_fired and now - self.fist_start >= FIST_HOLD:
                self.player.play_pause()
                self.fist_fired = True
                self.flash("PLAY / PAUSE")

        # --- Swipe tracks (index finger only) ---
        elif mode == "point":
            tip = lm[8]
            self.swipe_hist.append((now, tip.x, tip.y))
            self.swipe_hist = [h for h in self.swipe_hist if now - h[0] <= SWIPE_WINDOW]
            dx = self.swipe_hist[-1][1] - self.swipe_hist[0][1]
            dy = self.swipe_hist[-1][2] - self.swipe_hist[0][2]
            self.flash(f"Distance: {dx} of 0.25")
            if (abs(dx) > SWIPE_DIST and abs(dy) < abs(dx) * 0.7
                    and now - self.last_swipe > SWIPE_COOLDOWN):
                # Frame is mirrored, so dx<0 = your hand moved to your left
                if dx < 0:
                    self.player.next_track()
                    self.flash("NEXT TRACK")
                else:
                    self.player.previous_track()
                    self.flash("PREVIOUS TRACK")
                self.last_swipe = now
                self.swipe_hist.clear()
            

        # --- Volume (pinch + drag) ---
        elif mode == "pinch":
            y = lm[8].y
            if self.pinch_prev_y is None:
                self.pinch_prev_y = y
                self.vol_accum = 0.0
                if self.player.absolute_volume:
                    self.vol_start = self.player.get_volume()
            else:
                dy = self.pinch_prev_y - y          # up = positive
                self.pinch_prev_y = y
                if abs(dy) > VOLUME_DEADZONE:
                    self.vol_accum += dy * VOLUME_GAIN
                    if self.player.absolute_volume:
                        v = int(np.clip(self.vol_start + self.vol_accum, 0, 100))
                        self.player.set_volume(v)
                        self.flash(f"VOLUME {v}%")
                    else:
                        steps = int(self.vol_accum / 2)   # ~2% per OS volume step
                        if steps:
                            self.player.volume_steps(steps)
                            self.vol_accum -= steps * 2
                            self.flash("VOLUME UP" if steps > 0 else "VOLUME DOWN")


# ---------------- Main loop ----------------
def main():
    ensure_model_exists()
    cap = cv2.VideoCapture(0)
    detector = HandDetector()
    ctrl = GestureController(Player())
    print("Hold an open palm for 0.5s to ARM. Press 'q' in the camera window to quit.")

    while cap.isOpened():
        ok, frame = cap.read()
        if not ok:
            break
        frame = cv2.flip(frame, 1)
        lm = detector.process(frame, int(time.time() * 1000))
        ctrl.update(lm)

        h, w, _ = frame.shape
        if lm:
            for p in lm:
                cv2.circle(frame, (int(p.x * w), int(p.y * h)), 4, (255, 0, 0), cv2.FILLED)

        color = (0, 200, 0) if ctrl.armed else (0, 0, 255)
        cv2.putText(frame, "ARMED" if ctrl.armed else "DISARMED", (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
        if ctrl.palm_progress > 0:  # hold-to-toggle progress bar
            cv2.rectangle(frame, (10, 55), (210, 70), (255, 255, 255), 1)
            cv2.rectangle(frame, (10, 55), (10 + int(200 * ctrl.palm_progress), 70), color, cv2.FILLED)
        if time.time() - ctrl.status_time < 1.5:
            cv2.putText(frame, ctrl.status, (10, h - 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)

        cv2.imshow("Spotify Gesture Control", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

Hold an open palm for 0.5s to ARM. Press 'q' in the camera window to quit.


C:\Users\evanh\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Hold an open palm for 0.5s to ARM. Press 'q' in the camera window to quit.
